# Causal LM Summarization: Qwen / DeepSeek LoRA

Notebook nay chay them nhanh causal LM LoRA trong cung repo. Chay sau khi ViT5 baseline da co ket qua. Mac dinh chi chay Qwen2.5-1.5B LoRA; DeepSeek tat mac dinh.


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
from zipfile import ZipFile

PROJECT_NAME = 'pretrained-summarization'
REPO_URL = 'https://github.com/Anhnguyen0812/pretrained-summarization.git'
REFRESH_REPO = True
WORKING = Path('/kaggle/working')
WORKING_REPO = WORKING / PROJECT_NAME
OUTPUT_ROOT = WORKING / 'summarization_outputs'
DATA_DIR = Path('/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization')
TRAIN_FILE = DATA_DIR / 'train-00000-of-00001.parquet'
VALID_FILE = DATA_DIR / 'valid-00000-of-00001.parquet'

os.chdir(WORKING)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print('CMD:', cmd, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(cmd, shell=True, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines=[]
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code=process.wait()
    if check and code != 0:
        raise RuntimeError(f'Command failed with exit code {code}: {cmd}\nLast lines:\n{"".join(lines[-100:])}')
    return code

def is_repo(path):
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'vn_summarization').exists()

if REFRESH_REPO and WORKING_REPO.exists():
    shutil.rmtree(WORKING_REPO)
if not is_repo(WORKING_REPO):
    run(f'git clone --depth 1 {REPO_URL} {WORKING_REPO}', cwd=WORKING)
else:
    run('git pull --ff-only', cwd=WORKING_REPO, check=False)
repo = WORKING_REPO
run('git log --oneline -1', cwd=repo)


CMD: git clone --depth 1 https://github.com/Anhnguyen0812/pretrained-summarization.git /kaggle/working/pretrained-summarization
Cloning into '/kaggle/working/pretrained-summarization'...
CMD: git log --oneline -1
0d57955 Add causal LM LoRA training pipeline


0

In [2]:
os.chdir(repo)
run(f'{sys.executable} -m pip install -q --upgrade pip', cwd=repo)
run(f'{sys.executable} -m pip install -q --upgrade --force-reinstall --no-deps transformers==4.46.3 tokenizers==0.20.3', cwd=repo)
run(f'{sys.executable} -m pip install -q -e .', cwd=repo)
run(f'{sys.executable} -m pip show transformers tokenizers peft accelerate | sed -n "/Name: /p;/Version: /p"', cwd=repo, check=False)


CMD: /usr/bin/python3 -m pip install -q --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.1 MB/s eta 0:00:00
CMD: /usr/bin/python3 -m pip install -q --upgrade --force-reinstall --no-deps transformers==4.46.3 tokenizers==0.20.3
CMD: /usr/bin/python3 -m pip install -q -e .
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inc

0

In [3]:
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
run('nvidia-smi', check=False)
print('TRAIN_FILE:', TRAIN_FILE, TRAIN_FILE.exists())
print('VALID_FILE:', VALID_FILE, VALID_FILE.exists())
if not TRAIN_FILE.exists() or not VALID_FILE.exists():
    raise FileNotFoundError('Attach Kaggle dataset first.')

import torch
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('NUM_GPUS', NUM_GPUS)


CMD: nvidia-smi
Tue Jun  9 06:07:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------

In [4]:
BASE_OVERRIDES = [
    f'data.train_file={TRAIN_FILE}',
    f'data.valid_file={VALID_FILE}',
]
RUN_CONFIGS = {}

def _override_string(items):
    return ' '.join(f'--set {item}' for item in items)

def latest_checkpoint(run_dir):
    checkpoints = sorted(run_dir.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]) if p.name.split('-')[-1].isdigit() else -1)
    return checkpoints[-1] if checkpoints else None

def train_causal(run_name, config_path, overrides=None, overwrite=False):
    run_dir = OUTPUT_ROOT / run_name
    RUN_CONFIGS[run_name] = config_path
    if (run_dir / 'best' / 'adapter_config.json').exists() and not overwrite:
        print('SKIP train, adapter exists:', run_dir / 'best')
        return 0
    resume = []
    if run_dir.exists() and not overwrite:
        ckpt = latest_checkpoint(run_dir)
        if ckpt:
            print('RESUME', ckpt)
            resume = [f'training.resume_from_checkpoint={ckpt}']
        else:
            print('CLEAR incomplete run:', run_dir)
            shutil.rmtree(run_dir)
    final_overrides = BASE_OVERRIDES + [f'training.output_dir={run_dir}'] + resume + (overrides or [])
    train_cmd = f'-m vn_summarization.train_causal_lm --config {config_path} {_override_string(final_overrides)}'
    if NUM_GPUS >= 2:
        cmd = f'{sys.executable} -m accelerate.commands.launch --multi_gpu --num_processes {NUM_GPUS} --mixed_precision fp16 {train_cmd}'
    else:
        cmd = f'{sys.executable} -u {train_cmd}'
    return run(cmd, cwd=repo)

def eval_causal(eval_name, config_path, model_path, overrides=None, overwrite=False):
    eval_dir = OUTPUT_ROOT / eval_name
    if (eval_dir / 'validation_metrics.json').exists() and not overwrite:
        print('SKIP eval:', eval_dir / 'validation_metrics.json')
        return 0
    eval_dir.mkdir(parents=True, exist_ok=True)
    final_overrides = BASE_OVERRIDES + [f'training.output_dir={eval_dir}'] + (overrides or [])
    cmd = (
        f'{sys.executable} -u -m vn_summarization.evaluate_causal_lm '
        f'--config {config_path} --model_path {model_path} '
        f'--predictions_path {eval_dir / "predictions_valid.jsonl"} '
        f'{_override_string(final_overrides)}'
    )
    return run(cmd, cwd=repo)

def summarize_results():
    return run(f'{sys.executable} -u -m vn_summarization.summarize_results --root {OUTPUT_ROOT}', cwd=repo, check=False)


## Run Flags

Mac dinh: Qwen LoRA 1 epoch quick-ish full data. Neu can nhanh hon, dat `data.max_train_samples=3000`, `data.max_eval_samples=300` trong overrides.


In [5]:
RUN_QWEN = True
RUN_DEEPSEEK = False
OVERWRITE_RUNS = False
OVERWRITE_EVALS = False

COMMON_CAUSAL = [
    'training.num_train_epochs=1',
    'training.eval_steps=500',
    'training.save_steps=500',
    'training.logging_steps=100',
    'training.save_total_limit=2',
    'training.ddp_find_unused_parameters=false',
    'training.per_device_train_batch_size=1',
    'training.per_device_eval_batch_size=1',
    'training.gradient_accumulation_steps=8',
    'data.max_source_length=1024',
    'data.max_target_length=180',
    'data.max_length=1204',
]

EXPERIMENTS=[]
if RUN_QWEN:
    EXPERIMENTS.append({
        'name': 'qwen25_15b_lora_ep1_t4x2',
        'config': 'configs/qwen25_15b_lora.yaml',
        'overrides': COMMON_CAUSAL,
    })
if RUN_DEEPSEEK:
    EXPERIMENTS.append({
        'name': 'deepseek_r1_qwen15b_lora_ep1_t4x2',
        'config': 'configs/deepseek_r1_distill_qwen15b_lora.yaml',
        'overrides': COMMON_CAUSAL,
    })
print(json.dumps(EXPERIMENTS, indent=2))


[
  {
    "name": "qwen25_15b_lora_ep1_t4x2",
    "config": "configs/qwen25_15b_lora.yaml",
    "overrides": [
      "training.num_train_epochs=1",
      "training.eval_steps=500",
      "training.save_steps=500",
      "training.logging_steps=100",
      "training.save_total_limit=2",
      "training.ddp_find_unused_parameters=false",
      "training.per_device_train_batch_size=1",
      "training.per_device_eval_batch_size=1",
      "training.gradient_accumulation_steps=8",
      "data.max_source_length=1024",
      "data.max_target_length=180",
      "data.max_length=1204"
    ]
  }
]


## Train And Evaluate

Sau train, notebook evaluate ROUGE bang generation tren validation. Evaluation co the ton thoi gian vi causal LM generate cham hon seq2seq.


In [6]:
for exp in EXPERIMENTS:
    print('\n' + '='*100)
    print('TRAIN', exp['name'])
    train_causal(exp['name'], exp['config'], exp['overrides'], overwrite=OVERWRITE_RUNS)
    model_path = OUTPUT_ROOT / exp['name'] / 'best'
    print('EVAL', model_path)
    eval_causal(exp['name'] + '_eval', exp['config'], model_path, exp['overrides'], overwrite=OVERWRITE_EVALS)
    summarize_results()



TRAIN qwen25_15b_lora_ep1_t4x2
CMD: /usr/bin/python3 -m accelerate.commands.launch --multi_gpu --num_processes 2 --mixed_precision fp16 -m vn_summarization.train_causal_lm --config configs/qwen25_15b_lora.yaml --set data.train_file=/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/train-00000-of-00001.parquet --set data.valid_file=/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/valid-00000-of-00001.parquet --set training.output_dir=/kaggle/working/summarization_outputs/qwen25_15b_lora_ep1_t4x2 --set training.num_train_epochs=1 --set training.eval_steps=500 --set training.save_steps=500 --set training.logging_steps=100 --set training.save_total_limit=2 --set training.ddp_find_unused_parameters=false --set training.per_device_train_batch_size=1 --set training.per_device_eval_batch_size=1 --set training.gradient_accumulation_steps=8 --set data.max_source_length=1024 --set data.max_target_length=180 --set data.max_length=1204
The following values were not p

## Summarize And Export


In [7]:
summarize_results()
zip_path = WORKING / 'causal_lm_results.zip'
if zip_path.exists():
    zip_path.unlink()
keep_suffixes = {'.json', '.jsonl', '.csv', '.md', '.txt'}
files = [p for p in OUTPUT_ROOT.rglob('*') if p.is_file() and p.suffix in keep_suffixes]
with ZipFile(zip_path, 'w') as zf:
    for file in files:
        zf.write(file, file.relative_to(WORKING).as_posix())
print('ZIP', zip_path)
for file in sorted(files)[:80]:
    print(file)


CMD: /usr/bin/python3 -u -m vn_summarization.summarize_results --root /kaggle/working/summarization_outputs
# Result Summary

| run | model | lora | rouge1 | rouge2 | rougeL | gen_len | loss |
| --- | --- | --- | --- | --- | --- | --- | --- |
| qwen25_15b_lora_ep1_t4x2_eval |  |  | 67.9502 | 30.7444 | 36.2017 | 93.8762 |  |
| qwen25_15b_lora_ep1_t4x2 | Qwen/Qwen2.5-1.5B-Instruct | True |  |  |  |  | 0.5272 |

CSV: /kaggle/working/summarization_outputs/summary_results.csv
Best: /kaggle/working/summarization_outputs/best_run.json
ZIP /kaggle/working/causal_lm_results.zip
/kaggle/working/summarization_outputs/best_run.json
/kaggle/working/summarization_outputs/qwen25_15b_lora_ep1_t4x2/all_results.json
/kaggle/working/summarization_outputs/qwen25_15b_lora_ep1_t4x2/best/README.md
/kaggle/working/summarization_outputs/qwen25_15b_lora_ep1_t4x2/best/adapter_config.json
/kaggle/working/summarization_outputs/qwen25_15b_lora_ep1_t4x2/best/added_tokens.json
/kaggle/working/summarization_outputs/qw